In [4]:
import numpy as np
from scipy.optimize import linprog
def naive_parametric_simplex(
    H, h, a, b,
    m_min, m_max,
    n_grid=5000,
    eps=1e-6
):
    """
    Naive parametric LP by grid search over m using scipy.linprog (simplex).
    Returns list of (m_left, m_right, basis).
    Intervals are tight (no gaps).
    """
    ms = np.linspace(m_min, m_max, n_grid)
    bases = []

    for m in ms:
        c = a + m * b
        res = linprog(
            c,
            A_eq=H,
            b_eq=h,
            bounds=[(0, None)] * len(a),
            method="simplex"
        )

        if not res.success:
            bases.append(None)
            continue

        # simplex trả basis trực tiếp
        basis = tuple(sorted(res.basis))
        bases.append(basis)

    intervals = []
    start = 0

    for k in range(1, len(ms)):
        if bases[k] != bases[start]:
            # interval trước kết thúc tại ms[k]
            intervals.append((ms[start], ms[k], bases[start]))
            start = k

    intervals.append((ms[start], ms[-1], bases[start]))

    return intervals


In [5]:
EPS = 1e-9

def reduced_cost_affine(H, a, b, B, j):
    """
    rc_j(m) = alpha_j + beta_j * m
    """
    HB = H[:, B]
    invHB = np.linalg.inv(HB)

    aB = a[B]
    bB = b[B]
    Hj = H[:, j]

    alpha = a[j] - aB @ invHB @ Hj
    beta  = b[j] - bB @ invHB @ Hj
    return alpha, beta
def basis_interval(H, a, b, B, m_cur, m_max):
    """
    Tìm interval [m_cur, m_next] sao cho basis B còn tối ưu.
    """
    n = H.shape[1]
    m_next = m_max
    entering = None

    for j in range(n):
        if j in B:
            continue

        alpha, beta = reduced_cost_affine(H, a, b, B, j)

        # rc_j(m) >= 0
        if abs(beta) < EPS:
            if alpha < 0:
                return None, None
        else:
            root = -alpha / beta
            if root > m_cur + EPS:
                if root < m_next:
                    m_next = root
                    entering = j

    return m_next, entering
def simplex_pivot(H, h, B, j_enter):
    HB = H[:, B]
    invHB = np.linalg.inv(HB)

    d = invHB @ H[:, j_enter]
    xB = invHB @ h

    ratios = np.full(len(B), np.inf)
    for i in range(len(B)):
        if d[i] > EPS:
            ratios[i] = xB[i] / d[i]

    leave_idx = np.argmin(ratios)
    j_leave = B[leave_idx]

    B_new = B.copy()
    B_new[leave_idx] = j_enter
    return B_new, j_leave
    

def parametric_simplex(
    H, h, a, b,
    m_min, m_max,
    m0=None
):
    """
    Parametric simplex for c(m) = a + m b
    Returns list of (m_left, m_right, basis)
    """

    # --- khởi tạo m0 ---
    if m0 is None:
        m0 = m_min

    # --- solve LP tại m0 để lấy basis hợp lệ ---
    res = linprog(
        a + m0 * b,
        A_eq=H,
        b_eq=h,
        bounds=[(0,None)] * len(a),
        method="simplex"
    )

    if not res.success:
        raise RuntimeError("Initial LP infeasible")

    B = list(sorted(res.basis))
    rank = H.shape[0]
    assert len(B) == rank, "Basis size != rank(H)"

    intervals = []
    m_cur = m0

    while m_cur < m_max - EPS:
        m_next, entering = basis_interval(H, a, b, B, m_cur, m_max)

        if m_next is None:
            break

        intervals.append((m_cur, m_next, B.copy()))

        if entering is None or m_next >= m_max - EPS:
            break

        # pivot
        B, j_leave = simplex_pivot(H, h, B, entering)
        B = sorted(B)
        m_cur = m_next

    return intervals


In [6]:
# Constraint matrix (transportation)
H = np.array([
    [1,1,0,0],  # supply 1
    [0,0,1,1],  # supply 2
    [1,0,1,0],  # demand 1
    [0,1,0,1],  # demand 2
], dtype=float)

h = np.array([10, 15, 12, 13], dtype=float)
H = H[:-1, :]
h = h[:-1]
# Cost: c = a + m b
a = np.array([2, 4, 3, 1], dtype=float)
b = np.array([1,-1,0,2], dtype=float)

# Initial feasible basis (tree)
B = [0,1,2,3]  # all vars basic for simplicity

m_min = -10
m_max = 10


In [7]:
intervals = parametric_simplex(
    H, h, a, b,
    m_min=-20,
    m_max=20
)

for k,(ml,mr,B) in enumerate(intervals):
    print(f"Interval {k}: m in [{ml:.3f}, {mr:.3f}]")
    print(f"  Basis: {B}")

Interval 0: m in [-20.000, 1.000]
  Basis: [np.int64(0), np.int64(2), np.int64(3)]
Interval 1: m in [1.000, 20.000]
  Basis: [1, np.int64(2), np.int64(3)]


C:\Users\Asus\AppData\Local\Temp\ipykernel_29936\308006050.py:78: DeprecationWarning: `method='simplex'` is deprecated and will be removed in SciPy 1.11.0. Please use one of the HiGHS solvers (e.g. `method='highs'`) in new code.
  res = linprog(


In [8]:
intervals_naive = naive_parametric_simplex(
    H, h, a, b,
    m_min=-20,
    m_max=20,
    n_grid=5000
)

for k,(ml,mr,supp) in enumerate(intervals_naive):
    print(f"Interval {k}: m in [{ml:.3f},{mr:.3f}]")
    print(f"  Basis: {supp}")

C:\Users\Asus\AppData\Local\Temp\ipykernel_29936\4264009367.py:19: DeprecationWarning: `method='simplex'` is deprecated and will be removed in SciPy 1.11.0. Please use one of the HiGHS solvers (e.g. `method='highs'`) in new code.
  res = linprog(


Interval 0: m in [-20.000,1.004]
  Basis: (np.int64(0), np.int64(2), np.int64(3))
Interval 1: m in [1.004,20.000]
  Basis: (np.int64(1), np.int64(2), np.int64(3))


In [9]:
# 3x3 transportation
H = np.array([
    [1,1,1, 0,0,0, 0,0,0],   # supply 1
    [0,0,0, 1,1,1, 0,0,0],   # supply 2
    [0,0,0, 0,0,0, 1,1,1],   # supply 3
    [1,0,0, 1,0,0, 1,0,0],   # demand 1
    [0,1,0, 0,1,0, 0,1,0],   # demand 2
    [0,0,1, 0,0,1, 0,0,1],   # demand 3
], dtype=float)

h = np.array([20, 30, 25, 25, 20, 30], dtype=float)

# drop 1 redundant constraint
H = H[:-1, :]
h = h[:-1]

print("H shape:", H.shape)   # (5,9)

H shape: (5, 9)


In [10]:
# base cost
a = np.array([
    4, 6, 9,
    5, 4, 7,
    6, 3, 8
], dtype=float)

# parametric direction
b = np.array([
     2, -1,  0,
    -2,  1,  3,
     0,  2, -1
], dtype=float)


In [11]:
intervals_param = parametric_simplex(
    H, h, a, b,
    m_min=-10,
    m_max=10
)

print("=== Parametric simplex ===")
for k,(ml,mr,B) in enumerate(intervals_param):
    print(f"Interval {k}: m in [{ml:.3f},{mr:.3f}]")
    print(f"  Basis: {B}")


=== Parametric simplex ===
Interval 0: m in [-10.000,-2.000]
  Basis: [np.int64(0), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
Interval 1: m in [-2.000,0.000]
  Basis: [np.int64(0), 3, np.int64(5), np.int64(6), np.int64(7)]
Interval 2: m in [0.000,0.400]
  Basis: [np.int64(0), 3, np.int64(5), np.int64(7), 8]
Interval 3: m in [0.400,0.429]
  Basis: [np.int64(0), 3, 4, np.int64(5), 8]
Interval 4: m in [0.429,0.500]
  Basis: [np.int64(0), 2, 3, 4, 8]
Interval 5: m in [0.500,10.000]
  Basis: [1, 2, 3, 4, 8]


C:\Users\Asus\AppData\Local\Temp\ipykernel_29936\308006050.py:78: DeprecationWarning: `method='simplex'` is deprecated and will be removed in SciPy 1.11.0. Please use one of the HiGHS solvers (e.g. `method='highs'`) in new code.
  res = linprog(


In [12]:
intervals_naive = naive_parametric_simplex(
    H, h, a, b,
    m_min=-10,
    m_max=10,
    n_grid=8000
)

print("\n=== Naive (SciPy simplex) ===")
for k,(ml,mr,B) in enumerate(intervals_naive):
    print(f"Interval {k}: m in [{ml:.3f},{mr:.3f}]")
    print(f"  Basis: {B}")


C:\Users\Asus\AppData\Local\Temp\ipykernel_29936\4264009367.py:19: DeprecationWarning: `method='simplex'` is deprecated and will be removed in SciPy 1.11.0. Please use one of the HiGHS solvers (e.g. `method='highs'`) in new code.
  res = linprog(



=== Naive (SciPy simplex) ===
Interval 0: m in [-10.000,-1.999]
  Basis: (np.int64(0), np.int64(4), np.int64(5), np.int64(6), np.int64(7))
Interval 1: m in [-1.999,0.001]
  Basis: (np.int64(0), np.int64(3), np.int64(5), np.int64(6), np.int64(7))
Interval 2: m in [0.001,0.401]
  Basis: (np.int64(0), np.int64(3), np.int64(5), np.int64(7), np.int64(8))
Interval 3: m in [0.401,0.429]
  Basis: (np.int64(0), np.int64(3), np.int64(4), np.int64(5), np.int64(8))
Interval 4: m in [0.429,0.501]
  Basis: (np.int64(0), np.int64(2), np.int64(3), np.int64(4), np.int64(8))
Interval 5: m in [0.501,10.000]
  Basis: (np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(8))


-------------------------------
# Quadratic cost

In [13]:
EPS = 1e-9


# =====================================================
# Reduced cost quadratic: r_j(m) = alpha + beta m + gamma m^2
# =====================================================
def reduced_cost_quadratic(H, a, b, c, B, j):
    HB = H[:, list(B)]
    invHB = np.linalg.inv(HB)
    Hj = H[:, j]

    aB = a[list(B)]
    bB = b[list(B)]
    cB = c[list(B)]

    alpha = a[j] - aB @ invHB @ Hj
    beta  = b[j] - bB @ invHB @ Hj
    gamma = c[j] - cB @ invHB @ Hj

    return alpha, beta, gamma


# =====================================================
# Find smallest root > m0 of alpha + beta m + gamma m^2 = 0
# =====================================================
def next_root_after(alpha, beta, gamma, m0):
    roots = []

    if abs(gamma) < EPS:
        if abs(beta) > EPS:
            r = -alpha / beta
            if r > m0 + EPS:
                roots.append(r)
    else:
        D = beta * beta - 4 * gamma * alpha
        if D >= 0:
            sqrtD = np.sqrt(D)
            r1 = (-beta - sqrtD) / (2 * gamma)
            r2 = (-beta + sqrtD) / (2 * gamma)
            for r in (r1, r2):
                if r > m0 + EPS:
                    roots.append(r)

    return min(roots) if roots else None


# =====================================================
# Next breakpoint for current basis
# =====================================================
def next_breakpoint_quadratic(H, a, b, c, B, m0, m_max):
    m_next = m_max

    for j in range(H.shape[1]):
        if j in B:
            continue

        alpha, beta, gamma = reduced_cost_quadratic(H, a, b, c, B, j)

        # Must be optimal at m0
        if alpha + beta * m0 + gamma * m0 * m0 < -EPS:
            return None

        r = next_root_after(alpha, beta, gamma, m0)
        if r is not None and r < m_next:
            m_next = r

    return m_next


# =====================================================
# PARAMETRIC SIMPLEX – QUADRATIC (RESTART, CORRECT)
# =====================================================
def parametric_simplex_quadratic_restart(
    H, h, a, b, c,
    m_min, m_max,
    eps_restart=1e-6
):
    intervals = []
    m_cur = m_min

    while m_cur < m_max - EPS:

        # Solve LP at current m
        res = linprog(
            a + b * m_cur + c * (m_cur ** 2),
            A_eq=H,
            b_eq=h,
            bounds=[(0, None)] * len(a),
            method="simplex"
        )

        if not res.success:
            break

        B = tuple(sorted(res.basis))

        # Find next breakpoint
        m_next = next_breakpoint_quadratic(
            H, a, b, c, B, m_cur, m_max
        )

        if m_next is None or m_next <= m_cur + EPS:
            break

        intervals.append((m_cur, m_next, B))

        m_cur = m_next + eps_restart

    return intervals


# =====================================================
# Objective value from basis (for verification)
# =====================================================
def objective_from_basis(H, h, a, b, c, B, m):
    HB = H[:, list(B)]
    xB = np.linalg.solve(HB, h)
    x = np.zeros(H.shape[1])
    x[list(B)] = xB
    return (a + b*m + c*m*m) @ x

In [14]:
# Ma trận ràng buộc transportation 3x3
H = np.array([
    [1,1,1, 0,0,0, 0,0,0],   # supply 1
    [0,0,0, 1,1,1, 0,0,0],   # supply 2
    [0,0,0, 0,0,0, 1,1,1],   # supply 3
    [1,0,0, 1,0,0, 1,0,0],   # demand 1
    [0,1,0, 0,1,0, 0,1,0],   # demand 2
    [0,0,1, 0,0,1, 0,0,1],   # demand 3
], dtype=float)

h = np.array([20, 30, 25, 25, 20, 30], dtype=float)

# Drop 1 ràng buộc dư thừa
H = H[:-1, :]
h = h[:-1]

print("H shape:", H.shape)   # (5,9)


# ===============================
# 2. COST QUADRATIC
# c(m) = a + b*m + c*m^2
# ===============================

a = np.array([
    4, 6, 9,
    5, 4, 7,
    6, 3, 8
], dtype=float)

b = np.array([
     2, -1,  0,
    -2,  1,  3,
     0,  2, -1
], dtype=float)

c = np.array([
     0.5,  0.2, -0.1,
    -0.3,  0.4,  0.1,
     0.2, -0.2,  0.3
], dtype=float)


# ===============================
# 3. CHẠY PARAMETRIC SIMPLEX
# ===============================

intervals = parametric_simplex_quadratic_restart(
    H, h,
    a, b, c,
    # m_start=-10.0,
    # m_end=10.0,
    m_min=0.0 +1e-8,
    m_max=10.0
)


# ===============================
# 4. IN KẾT QUẢ
# ===============================

print("\n=== PARAMETRIC SIMPLEX (QUADRATIC, LOCAL) ===")
for k, (ml, mr, B) in enumerate(intervals):
    print(f"Interval {k}: m in [{ml:.3f}, {mr:.3f}]")
    print(f"  Basis: {B}")

H shape: (5, 9)

=== PARAMETRIC SIMPLEX (QUADRATIC, LOCAL) ===
Interval 0: m in [0.000, 0.405]
  Basis: (np.int64(0), np.int64(3), np.int64(5), np.int64(7), np.int64(8))
Interval 1: m in [0.405, 0.430]
  Basis: (np.int64(2), np.int64(3), np.int64(5), np.int64(7), np.int64(8))
Interval 2: m in [0.430, 0.564]
  Basis: (np.int64(2), np.int64(3), np.int64(4), np.int64(7), np.int64(8))
Interval 3: m in [0.564, 4.436]
  Basis: (np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(8))
Interval 4: m in [4.436, 5.820]
  Basis: (np.int64(2), np.int64(3), np.int64(4), np.int64(7), np.int64(8))
Interval 5: m in [5.820, 10.000]
  Basis: (np.int64(2), np.int64(3), np.int64(5), np.int64(7), np.int64(8))


C:\Users\Asus\AppData\Local\Temp\ipykernel_29936\428106229.py:84: DeprecationWarning: `method='simplex'` is deprecated and will be removed in SciPy 1.11.0. Please use one of the HiGHS solvers (e.g. `method='highs'`) in new code.
  res = linprog(


In [15]:
def basic_solution_from_basis(H, h, B, n):
    x = np.zeros(n)
    HB = H[:, list(B)]
    xB = np.linalg.solve(HB, h)
    x[list(B)] = xB
    return x
def objective_value(a, b, c, x, m):
    return (a + b*m + c*(m**2)) @ x

In [16]:
print("\n=== KIỂM TRA GIÁ TRỊ HÀM MỤC TIÊU ===")

for k, (ml, mr, B) in enumerate(intervals):
    # chọn m đại diện (giữa interval)
    m_test = 0.5 * (ml + mr)

    # nghiệm từ basis parametric simplex
    x_basis = basic_solution_from_basis(H, h, B, len(a))
    val_basis = objective_value(a, b, c, x_basis, m_test)

    # nghiệm chuẩn từ linprog
    res = linprog(
        a + b*m_test + c*(m_test**2),
        A_eq=H,
        b_eq=h,
        bounds=[(0, None)] * len(a),
        method="simplex"
    )

    if not res.success:
        print(f"Interval {k}: LP fail at m={m_test:.4f}")
        continue

    val_lp = (a + b*m_test + c*(m_test**2)) @ res.x

    print(f"Interval {k}= {ml, mr}: m = {m_test:.4f}")
    print(f"  Basis value = {val_basis:.10f}")
    print(f"  LP value    = {val_lp:.10f}")
    print(f"  |diff|      = {abs(val_basis - val_lp):.3e}")
    print(f"  Basis of parametric simplex: {tuple(int(x) for x in B)}")
    print(f"  LP solution: {res.basis}\n")



=== KIỂM TRA GIÁ TRỊ HÀM MỤC TIÊU ===
Interval 0= (1e-08, np.float64(0.40512483795332704)): m = 0.2026
  Basis value = 408.7075074094
  LP value    = 408.7075074094
  |diff|      = 0.000e+00
  Basis of parametric simplex: (0, 3, 5, 7, 8)
  LP solution: [0 3 5 7 8]

Interval 1= (np.float64(0.405125837953327), np.float64(0.4295176683940216)): m = 0.4173
  Basis value = 437.9971893748
  LP value    = 437.9971893748
  |diff|      = 0.000e+00
  Basis of parametric simplex: (2, 3, 5, 7, 8)
  LP solution: [2 3 5 7 8]

Interval 2= (np.float64(0.4295186683940216), np.float64(0.5635083268962915)): m = 0.4965
  Basis value = 435.7382201588
  LP value    = 435.7382201588
  |diff|      = 0.000e+00
  Basis of parametric simplex: (2, 3, 4, 7, 8)
  LP solution: [2 3 4 7 8]

Interval 3= (np.float64(0.5635093268962915), np.float64(4.436491673103708)): m = 2.5000
  Basis value = 295.6249687500
  LP value    = 295.6249687500
  |diff|      = 0.000e+00
  Basis of parametric simplex: (1, 2, 3, 4, 8)
  LP so

C:\Users\Asus\AppData\Local\Temp\ipykernel_29936\3726714658.py:12: DeprecationWarning: `method='simplex'` is deprecated and will be removed in SciPy 1.11.0. Please use one of the HiGHS solvers (e.g. `method='highs'`) in new code.
  res = linprog(


In [17]:
def find_valid_basis(H, x_nonzero_indices):
    """Tìm một tập chỉ số sao cho ma trận H[:, basis] là khả nghịch (Full Rank)"""
    num_constraints = H.shape[0]
    basis = list(x_nonzero_indices)
    
    # Sử dụng khử Gauss (hoặc QR) để tìm các cột độc lập tuyến tính bổ sung
    # Ở đây dùng phương pháp đơn giản: thử từng cột
    for i in range(H.shape[1]):
        if len(basis) == num_constraints:
            break
        if i not in basis:
            test_basis = basis + [i]
            if np.linalg.matrix_rank(H[:, test_basis]) == len(test_basis):
                basis.append(i)
    return sorted(basis)

def solve_quadratic_roots(a, b, c, current_z, z_max):
    """Tìm nghiệm thực nhỏ nhất của az^2 + bz + c = 0 trong khoảng (current_z, z_max]"""
    eps = 1e-7
    roots = []
    
    if abs(a) < 1e-10: # Tuyến tính: bz + c = 0
        if abs(b) > 1e-10:
            roots.append(-c / b)
    else: # Bậc hai
        delta = b**2 - 4*a*c
        if delta >= 0:
            sqrt_delta = np.sqrt(delta)
            roots.append((-b + sqrt_delta) / (2*a))
            roots.append((-b - sqrt_delta) / (2*a))
            
    valid_roots = [r for r in roots if r > current_z + eps and r <= z_max + eps]
    return min(valid_roots) if valid_roots else None

def parametric_simplex_quadratic(H, h, u, v, w, z_min, z_max):
    H = np.array(H, dtype=float)
    h = np.array(h, dtype=float)
    u, v, w = np.array(u), np.array(v), np.array(w)
    print("Shape:", H.shape, h.shape, u.shape, v.shape, w.shape)
    num_constraints, num_vars = H.shape
    current_z = z_min
    results = []

    # 1. Tìm lời giải ban đầu tại z_min
    c_start = u + v * z_min + w * (z_min**2)
    # Sử dụng method='highs' để có độ chính xác tốt hơn
    res = linprog(c_start, A_eq=H, b_eq=h, method='simplex')
    
    if not res.success:
        return [] # Trả về rỗng nếu không có lời giải khả thi

    # Xác định các biến cơ sở ban đầu (đảm bảo ma trận vuông và độc lập tuyến tính)
    nonzero_x = np.where(res.x > 1e-8)[0]
    basis_indices = res.basis#find_valid_basis(H, nonzero_x)

    while current_z <= z_max + 1e-9:
        B = H[:, basis_indices]
        try:
            B_inv = np.linalg.inv(B)
        except np.linalg.LinAlgError:
            # Nếu xảy ra lỗi ma trận suy biến, cố gắng tìm basis khác từ x hiện tại
            break

        # 2. Tính hệ số Reduced Cost: RC(z) = A*z^2 + B*z + C
        cb_u, cb_v, cb_w = u[basis_indices], v[basis_indices], w[basis_indices]
        
        # Công thức: rc = c - cB * B^-1 * H
        print("pi_u = cb_u @ B_inv", cb_u.shape, B_inv.shape)

        pi_u = cb_u @ B_inv
        pi_v = cb_v @ B_inv
        pi_w = cb_w @ B_inv
        
        rc_u = u - pi_u @ H
        rc_v = v - pi_v @ H
        rc_w = w - pi_w @ H

        # 3. Tìm z_next (điểm mà một RC biến ngoài cơ sở trở thành < 0)
        next_z = z_max
        entering_var = None
        
        for i in range(num_vars):
            if i in basis_indices:
                continue
            
            # Giải phương trình RC_i(z) = 0
            root = solve_quadratic_roots(rc_w[i], rc_v[i], rc_u[i], current_z, z_max)
            if root is not None and root < next_z:
                next_z = root
                entering_var = i

        # Lưu lại khoảng ổn định
        results.append((current_z, next_z, list(basis_indices)))

        if entering_var is None or next_z >= z_max:
            break

        # 4. Thực hiện Pivot (Rời khỏi cơ sở)
        # Tính hướng di chuyển d = B^-1 * H_entering
        d = B_inv @ H[:, entering_var]
        x_B = B_inv @ h
        
        # Tỷ số Min-ratio để tìm biến rời khỏi cơ sở
        min_ratio = float('inf')
        leaving_idx_in_basis = -1
        for i in range(len(d)):
            if d[i] > 1e-9:
                ratio = x_B[i] / d[i]
                if ratio < min_ratio:
                    min_ratio = ratio
                    leaving_idx_in_basis = i
        
        if leaving_idx_in_basis == -1: # Không tìm được biến rời đi (unbounded)
            break
            
        basis_indices[leaving_idx_in_basis] = entering_var
        current_z = next_z

    return results

In [18]:
# Ma trận ràng buộc transportation 3x3
H = np.array([
    [1,1,1, 0,0,0, 0,0,0],   # supply 1
    [0,0,0, 1,1,1, 0,0,0],   # supply 2
    [0,0,0, 0,0,0, 1,1,1],   # supply 3
    [1,0,0, 1,0,0, 1,0,0],   # demand 1
    [0,1,0, 0,1,0, 0,1,0],   # demand 2
    [0,0,1, 0,0,1, 0,0,1],   # demand 3
], dtype=float)

h = np.array([20, 30, 25, 25, 20, 30], dtype=float)

# Drop 1 ràng buộc dư thừa
H = H[:-1, :]
h = h[:-1]


# ===============================
# 2. COST QUADRATIC
# c(m) = a + b*m + c*m^2
# ===============================

a = np.array([
    4, 6, 9,
    5, 4, 7,
    6, 3, 8
], dtype=float)

b = np.array([
     2, -1,  0,
    -2,  1,  3,
     0,  2, -1
], dtype=float)

c = np.array([
     0.5,  0.2, -0.1,
    -0.3,  0.4,  0.1,
     0.2, -0.2,  0.3
], dtype=float)


# ===============================
# 3. CHẠY PARAMETRIC SIMPLEX
# ===============================
# parametric_simplex_quadratic(H, h, u, v, w, z_min, z_max)
intervals = parametric_simplex_quadratic(
    H, h,
    a, b, c,
    # m_start=-10.0,
    # m_end=10.0,
    z_min=-10.0,
    z_max=10.0
)


# ===============================
# 4. IN KẾT QUẢ
# ===============================
# print(intervals)
print("\n=== PARAMETRIC SIMPLEX (QUADRATIC, LOCAL) ===")
for k, (ml, mr, B) in enumerate(intervals):
    print(f"Interval {k}: m in [{ml:.3f}, {mr:.3f}]")
    print(f"  Basis: {B}")

Shape: (5, 9) (5,) (9,) (9,) (9,)
pi_u = cb_u @ B_inv (5,) (5, 5)
pi_u = cb_u @ B_inv (5,) (5, 5)
pi_u = cb_u @ B_inv (5,) (5, 5)
pi_u = cb_u @ B_inv (5,) (5, 5)
pi_u = cb_u @ B_inv (5,) (5, 5)
pi_u = cb_u @ B_inv (5,) (5, 5)
pi_u = cb_u @ B_inv (5,) (5, 5)
pi_u = cb_u @ B_inv (5,) (5, 5)

=== PARAMETRIC SIMPLEX (QUADRATIC, LOCAL) ===
Interval 0: m in [-10.000, -7.405]
  Basis: [np.int64(2), np.int64(3), np.int64(5), np.int64(6), np.int64(7)]
Interval 1: m in [-7.405, -0.000]
  Basis: [np.int64(0), np.int64(3), np.int64(5), np.int64(6), np.int64(7)]
Interval 2: m in [-0.000, 0.405]
  Basis: [np.int64(0), np.int64(3), np.int64(5), np.int64(8), np.int64(7)]
Interval 3: m in [0.405, 0.430]
  Basis: [np.int64(2), np.int64(3), np.int64(5), np.int64(8), np.int64(7)]
Interval 4: m in [0.430, 0.564]
  Basis: [np.int64(2), np.int64(3), np.int64(4), np.int64(8), np.int64(7)]
Interval 5: m in [0.564, 4.436]
  Basis: [np.int64(2), np.int64(3), np.int64(4), np.int64(8), np.int64(1)]
Interval 6: m i

C:\Users\Asus\AppData\Local\Temp\ipykernel_29936\3526841248.py:47: DeprecationWarning: `method='simplex'` is deprecated and will be removed in SciPy 1.11.0. Please use one of the HiGHS solvers (e.g. `method='highs'`) in new code.
  res = linprog(c_start, A_eq=H, b_eq=h, method='simplex')


In [19]:
print("\n=== KIỂM TRA GIÁ TRỊ HÀM MỤC TIÊU ===")

for k, (ml, mr, B) in enumerate(intervals):
    # chọn m đại diện (giữa interval)
    m_test = 0.5 * (ml + mr)

    # nghiệm từ basis parametric simplex
    x_basis = basic_solution_from_basis(H, h, B, len(a))
    val_basis = objective_value(a, b, c, x_basis, m_test)

    # nghiệm chuẩn từ linprog
    res = linprog(
        a + b*m_test + c*(m_test**2),
        A_eq=H,
        b_eq=h,
        bounds=[(0, None)] * len(a),
        method="simplex"
    )

    if not res.success:
        print(f"Interval {k}: LP fail at m={m_test:.4f}")
        continue

    val_lp = (a + b*m_test + c*(m_test**2)) @ res.x

    print(f"Interval {k}= {ml, mr}: m = {m_test:.4f}")
    print(f"  Basis value = {val_basis:.10f}")
    print(f"  LP value    = {val_lp:.10f}")
    print(f"  |diff|      = {abs(val_basis - val_lp):.3e}")
    print(f"  Basis of parametric simplex: {tuple(int(x) for x in B)}")
    print(f"  LP solution: {res.basis}\n")



=== KIỂM TRA GIÁ TRỊ HÀM MỤC TIÊU ===
Interval 0= (-10.0, np.float64(-7.405124837953327)): m = -8.7026
  Basis value = -578.4227991311
  LP value    = -578.4227991311
  |diff|      = 0.000e+00
  Basis of parametric simplex: (2, 3, 5, 6, 7)
  LP solution: [2 3 5 6 7]

Interval 1= (np.float64(-7.405124837953327), np.float64(-0.0)): m = -3.7026
  Basis value = -112.3459265618
  LP value    = -112.3459265618
  |diff|      = 0.000e+00
  Basis of parametric simplex: (0, 3, 5, 6, 7)
  LP solution: [0 3 5 6 7]

Interval 2= (np.float64(-0.0), np.float64(0.40512483795332704)): m = 0.2026
  Basis value = 408.7075066922
  LP value    = 408.7075066922
  |diff|      = 0.000e+00
  Basis of parametric simplex: (0, 3, 5, 8, 7)
  LP solution: [0 3 5 7 8]

Interval 3= (np.float64(0.40512483795332704), np.float64(0.4295176683940216)): m = 0.4173
  Basis value = 437.9971941740
  LP value    = 437.9971941740
  |diff|      = 0.000e+00
  Basis of parametric simplex: (2, 3, 5, 8, 7)
  LP solution: [2 3 5 7 8]

C:\Users\Asus\AppData\Local\Temp\ipykernel_29936\3726714658.py:12: DeprecationWarning: `method='simplex'` is deprecated and will be removed in SciPy 1.11.0. Please use one of the HiGHS solvers (e.g. `method='highs'`) in new code.
  res = linprog(
